# Sanity 05 — the headline table

Mirrors `scripts/05_train_downstream.py`. Stage 05 joins embeddings back to the pinned benchmark
rows (exact key match) and trains: the 13-raw-feature baseline (their recipe), their-protocol
PCA64+XGB on our embedding, and our no-PCA readouts. Here: read the persisted metrics, rebuild
the cross-scale table, and re-verify the join yourself.

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))   # template root (src/)
BASE = os.environ.get("FM_BASE", "/mnt/user_storage/transaction-fm-v2")
SCALE = "full"                               # flip to "xl" / "xxl" to inspect those runs

In [ ]:
rows = []
for sc in ("full", "xl", "xxl"):
    p = f"{BASE}/downstream/{sc}/benchmark_metrics.json"
    if not os.path.exists(p):
        print(f"({sc}: not run yet)"); continue
    m = json.load(open(p))
    for name, r in m["results"].items():
        rows.append({"scale": sc, "model": name, "roc_auc": round(r["auc_roc"], 4), "ap": round(r["ap"], 4)})
    ref = m["nvidia_reference"]
t = pd.DataFrame(rows).pivot(index="model", columns="scale", values="ap")
print("NVIDIA published: baseline", ref["baseline"], "fusion", ref["combined"])
t

In [ ]:
# Re-verify the join with your own hands: every benchmark row should find exactly one embedding.
import pyarrow.dataset as pads
bench = pd.read_parquet(f"{BASE}/raw/full/benchmark.parquet")
bench["_amt_cents"] = np.round(bench["Amount"].to_numpy() * 100).astype(np.int64)
keys = pads.dataset(f"{BASE}/embeddings/full", format="parquet").to_table(
    columns=["card_id", "raw_ts", "raw_amount"]).to_pandas()
keys["_amt_cents"] = np.round(keys["raw_amount"].to_numpy() * 100).astype(np.int64)
keys = keys.rename(columns={"raw_ts": "_ts"}).drop_duplicates(["card_id", "_ts", "_amt_cents"])
j = bench.drop_duplicates(["card_id", "_ts", "_amt_cents"]).merge(keys, on=["card_id", "_ts", "_amt_cents"])
print(f"joined {len(j):,} / {len(bench):,} benchmark rows ({len(j)/len(bench):.2%})")

In [ ]:
# The lifts, computed from the artifacts (not copied from a doc).
base = t.loc["baseline"]
for name in t.index:
    if name != "baseline":
        print(f"{name:>18}: " + "  ".join(
            f"{sc}={t.loc[name, sc]/base[sc]-1:+.1%}" for sc in t.columns if pd.notna(t.loc[name, sc])))

**What to check:** the baseline row ≈ 0.142 AP at every scale (the gate — same rows, same
recipe every run); embed_xgb rising with context (0.258 → 0.273 → xxl); the join at 100.00%.
The controls live alongside: `downstream/full_probe/probe_metrics_shuffled_seed0.json`
(shuffled-label sanity) and `downstream/full_velocity/velocity_metrics.json` (velocity bar).